In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
import numpy as np
import pandas as pd
import pandas as pd
from prophet import Prophet
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, LabelEncoder,RobustScaler,StandardScaler
import numpy as np
from itertools import product
from sklearn.preprocessing import LabelEncoder
import datetime
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
from prophet.diagnostics import cross_validation, performance_metrics
from sklearn.model_selection import ParameterGrid
import json
from prophet.serialize import model_to_json
import os
import sys
import pickle
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error,mean_absolute_percentage_error,r2_score
from xgboost import XGBRegressor
import xgboost
import joblib
from prophet.plot import plot_cross_validation_metric
import random
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 20)

In [ ]:
def read_data(file_path):
    data=pd.read_excel(file_path)
    data['periode'] = pd.to_datetime(data['periode'], format='%b-%Y')
    non_kck_banda = data.loc[(data['kanwil'] != 'K C K') & (data['kanwil'] != 'BANDA ACEH')]
    non_kck_banda['amount']=non_kck_banda['amount']
    return non_kck_banda.drop(columns=['kodekw','Unnamed: 0']).reset_index(drop=True)

In [ ]:
def metrics(predicted, actual):
    RMSE  = round(np.sqrt(mean_squared_error(predicted, actual)),3)
    mae = mean_absolute_error(predicted, actual)
    mape = mean_absolute_percentage_error(predicted, actual)
    r2 = r2_score(predicted, actual)
    return {'RMSE':RMSE, 'MAE':mae,'MAPE':mape, 'R2':r2}

In [ ]:
def Evaluate_Model(actual, predicted,type):
    forecast = actual[['ds','y']]
    forecast = forecast.set_index('ds')
    forecast.rename(columns={'y':'Actual'}, inplace=True)
    forecast['Predicted'] = predicted['yhat'].values
    if(type=='forecast'):
        return forecast
    elif(type=='test'):
        forecast_tmp=forecast.loc[(forecast.index>'2024-03-01')&(forecast.index<='2024-06-01')]
    elif(type=='train'):
        forecast_tmp=forecast.loc[forecast.index<='2024-03-01']
    KPI = metrics(forecast_tmp['Predicted'], forecast_tmp['Actual'])
    print("TYPE: ",type)
    print("\nRMSE : {}   MAE : {}   R² : {}   MAPE : {}".format(KPI['RMSE'], KPI['MAE'], KPI['R2'], KPI['MAPE']))
    return forecast

In [ ]:
def dump_model(model,file_path,scalers_numerical_train,scalers_categorical_train,log_col,coefficients_df_sorted):
    data = {
        'model': model,
        'scalers_numerical_train': scalers_numerical_train,
        'scalers_categorical_train':scalers_categorical_train,
        'scalers_log_col':log_col,
        'coefficients_df_sorted':coefficients_df_sorted
    }
    with open (file_path,'wb') as f:
        pickle.dump(data,f)
def load_model(file_path):
    with open(file_path, 'rb') as f:
        loaded_data = pickle.load(f)
    model = loaded_data['model']
    scalers_num = loaded_data['scalers_numerical_train']
    scalers_cat = loaded_data['scalers_categorical_train']
    list_scaled_log_col=loaded_data['scalers_log_col']
    coefficients_df_sorted=loaded_data['coefficients_df_sorted']
    return model, scalers_num,scalers_cat,list_scaled_log_col,coefficients_df_sorted

In [ ]:
def detect_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    Q2 = df[column].quantile(0.5)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    lower_bound = max(lower_bound, 0)
    upper_bound = Q3 + 1.5 * IQR
    mean=Q2
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers, lower_bound, upper_bound,mean
def replace_outliers_with_bound(df, column, lower_bound, upper_bound):
    df[column] = df[column].apply(lambda x: upper_bound if x > upper_bound else lower_bound if x < lower_bound else x)
    return df
def remove_outliers(df, column, lower_bound, upper_bound):
    df_filtered = df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]
    return df_filtered
def is_corona_period(ds):
    date = pd.to_datetime(ds)
    return (date.year == 2020 or date.year == 2021)


In [ ]:
df_all=read_data('../data/Data s,p until des 2025/pinjaman forecast (until des 2025).xlsx')

In [ ]:
df_all_amount=df_all[['periode','kanwil','segmen','produk','amount']]
combinations = df_all_amount[['kanwil', 'segmen', 'produk']].drop_duplicates()
combinations = list(combinations.itertuples(index=False, name=None))

In [ ]:
all_data_prophet=pd.DataFrame()
for comb in combinations:
    print(f"Processing {comb}")
    df_0_0_0=df_all_amount.loc[(df_all_amount['kanwil']==comb[0])&(df_all_amount['produk']==comb[2])&(df_all_amount['segmen']==comb[1])]
    df_0_0_0['post_corona'] = df_0_0_0['periode'].apply(is_corona_period)
    df_0_0_0['pre_corona'] = ~df_0_0_0['periode'].apply(is_corona_period)

    # if(comb[3]=='KUPEDES'):
    #     df_0_0_0_train=df_0_0_0.loc[(df_0_0_0['periode']>='2022-01-01')&(df_0_0_0['periode']<='2024-03-01')]
    #     df_0_0_0_test=df_0_0_0.loc[(df_0_0_0['periode']>'2024-03-01')]
        # df_0_0_0_test=df_0_0_0.loc[(df_0_0_0['periode']>'2024-03-01')&(df_0_0_0['periode']<='2024-06-01')]
        # df_0_0_0_forecast=df_0_0_0.loc[(df_0_0_0['periode']>'2024-06-01')]
    # else:
        # df_0_0_0_test=df_0_0_0.loc[(df_0_0_0['periode']>'2024-03-01')&(df_0_0_0['periode']<='2024-06-01')]
        # df_0_0_0_forecast=df_0_0_0.loc[(df_0_0_0['periode']>'2024-06-01')]
        # df_0_0_0_test=df_0_0_0.loc[(df_0_0_0['periode']>'2024-03-01')&(df_0_0_0['periode']<='2024-06-01')]
        # df_0_0_0_forecast=df_0_0_0.loc[(df_0_0_0['periode']>'2024-06-01')]

    df_0_0_0_train=df_0_0_0.loc[df_0_0_0['periode']<='2024-03-01']
    df_0_0_0_test=df_0_0_0.loc[(df_0_0_0['periode']>'2024-03-01')&(df_0_0_0['periode']<='2024-06-01')]
    df_0_0_0_forecast=df_0_0_0.loc[(df_0_0_0['periode']>'2024-06-01')]



    df_0_0_0_train.rename(columns={'amount':'y','periode':'ds'}, inplace=True)
    df_0_0_0_train['y']=df_0_0_0_train['y']/10e9

    df_0_0_0_test.rename(columns={'amount':'y','periode':'ds'}, inplace=True)
    df_0_0_0_test['y']=df_0_0_0_test['y']/10e9

    df_0_0_0_forecast.rename(columns={'amount':'y','periode':'ds'}, inplace=True)
    df_0_0_0_forecast['y']=df_0_0_0_forecast['y']/10e9

    outliers_amount, lower_amount, upper_amount,mean_amount = detect_outliers_iqr(df_0_0_0_train.copy(), 'y')
    df_0_0_0_train = replace_outliers_with_bound(df_0_0_0_train.copy(), 'y', lower_amount, upper_amount)
    df_0_0_0_test = replace_outliers_with_bound(df_0_0_0_test.copy(), 'y', lower_amount, upper_amount)



    model = Prophet()
    model.add_seasonality(name='monthly-pre-corona', 
                    period=30, 
                    prior_scale = 0.1,
                    fourier_order=5, 
                    condition_name='pre_corona')

    model.add_seasonality(name='monthly-post-corona',
                    period=30, 
                    prior_scale = 0.1,
                    fourier_order=5,
                    condition_name='post_corona')
    model.fit(df_0_0_0_train.drop(columns=['kanwil','segmen','produk']))
    pred=model.predict(df_0_0_0_train.drop(columns=['kanwil','y','segmen','produk']))
    pred_test=model.predict(df_0_0_0_test.drop(columns=['kanwil','y','segmen','produk']))
    pred_forecast=model.predict(df_0_0_0_forecast.drop(columns=['kanwil','y','segmen','produk']))

    pred['yhat'] = pred['yhat'].apply(lambda x: lower_amount if x < 0 else x)
    pred_test['yhat'] = pred_test['yhat'].apply(lambda x: lower_amount if x < 0 else x)
    pred_forecast['yhat'] = pred_forecast['yhat'].apply(lambda x: lower_amount if x < 0 else x)

    forecast=Evaluate_Model(df_0_0_0_train, pred,'train')
    forecast_test=Evaluate_Model(df_0_0_0_test, pred_test,'test')
    forecast_forecast=Evaluate_Model(df_0_0_0_forecast, pred_forecast,'forecast')

    forecast['Partition'] = 'TRAIN'
    forecast_test['Partition'] = 'TEST'
    forecast_forecast['Partition']='FORECAST'

    combined_forecast = pd.concat([forecast, forecast_test,forecast_forecast])
    combined_forecast['Kanwil']=comb[0]
    combined_forecast['Segmen']=comb[1]
    combined_forecast['Product']=comb[2]
    combined_forecast['Method']='PROPHET'
    combined_forecast.rename(columns={'Actual':'Amount','Predicted':'Amount_Predict','ds':'Date'}, inplace=True)
    dump_model(model,f'../results/v5/pinjaman/prophet_xgb/dump_model/prophet/{comb[0]}_{comb[1]}_{comb[2]}.pkl',None,None,None,None)
    plt.figure(figsize=(10, 6))
    plt.plot(combined_forecast.index, combined_forecast['Amount'], label='Actual', marker='o')
    plt.plot(combined_forecast.index, combined_forecast['Amount_Predict'], label='Predicted', marker='x')
    plt.xlabel('Date')
    plt.ylabel('Values')
    plt.title(f'Actual vs Predicted for {comb[0]} - {comb[1]} - {comb[2]}')
    plt.legend()    
    plt.tight_layout()
    plot_filename = f"../visualization/new/pinjaman/prophet/plot_{comb[0]}_{comb[1]}_{comb[2]}.png"
    plt.savefig(plot_filename)
    plt.show()
    print(f"Plot saved as {plot_filename}")
    all_data_prophet=pd.concat([all_data_prophet,combined_forecast])
all_data_prophet.to_excel(f'../results/v5/pinjaman/prophet_xgb/final_results_prophet/all_results_pinjaman.xlsx')

# XGBoost

In [ ]:
static_columns_default=['periode','kanwil','segmen','produk','amount']
static_columns_pinjaman_rasio_fin_ro_level=['Yield','FTP Pinjaman','Credit Cost','ROA after tax']
static_columns_pinjaman_makro_nas_level=['Ekspor_nasional','Impor_nasional','Ekspor_Impor_nasional','PDB_nasional','Inflasi Inti_nasional',
                                         'BI Rate_nasional','10 Year Yield_nasional','PDB Dunia_nasional',
                                         'FFR_nasional','Yield UST10Y_nasional','Indeks Harga Komoditas_nasional',
                                         'Harga Minyak Dunia_nasional','Harga Batubara_nasional','Harga CPO_nasional',
                                         'Harga Bijih Besi_nasional','PDB Riil_nasional','Konsumsi Swasta Riil_nasional',
                                         'Konsumsi Pemerintah Riil_nasional','Investasi Riil_nasional','Ekspor Riil_nasional', 
                                         'Impor Riil_nasional','Inflasi, akhir periode_nasional','BI Rate, akhir periode_nasional',
                                         'BI 7-DRR, akhir periode_nasional','Nilai Tukar_nasional','Tingkat Pengangguran_nasional',
                                         'Yield SBN 1YR_nasional','Yield SBN 5YR_nasional','Yield SBN 10YR_nasional',
                                         'IHSG_nasional','IHPR_nasional']
static_columns_pinjaman_makro_ro_level=['Inflasi','Ekspor','Impor','pinjaman Industry','Pinjaman Industry','PDRB RO']
static_columns_pinjaman_merchant=['Merchant EDC & Online', 'Merchant QRIS','Sales Volume EDC Merchant Only', 
                                  'Sales Volume QRIS','Transaksi EDC Merchant Only', 'Transaksi Merchant QRIS']
static_columns_rm=['RM - Konsumer Briguna',
       'RM - Konsumer KPR', 'RM - Menengah', 'RM - SME']
static_columns_rasio_fin_nas_level=['BOPO_nasional', 'CASA_nasional','Credit Cost_nasional','LDR_nasional','LAR_nasional', 'NIM_nasional',
                                    'NPL (Gross)_nasional','Recovery_nasional', 'Recovery Rate_nasional',
                                    'ROA before tax_nasional','ROE B/S_nasional', 'Tier1 CAR_nasional', 
                                    'Total CAR_nasional','Biaya Hadiah (dalam Miliar)_nasional']

In [ ]:
def process_data_numeric_xgb(df,method):
    scalers={}
    list_of_col_log_scaled=[]
    if(method==None):
        return scalers,df,list_of_col_log_scaled
    for col in df.columns:
        if(method=='Standard'):
            scalers[col]=StandardScaler()
            df[col]=scalers[col].fit_transform(df[col].values.reshape(-1, 1))
        elif(method=='Robust'):
            scalers[col]=RobustScaler()
            df[col]=scalers[col].fit_transform(df[col].values.reshape(-1, 1))
        elif(method=='MinMax'):
            scalers[col]=MinMaxScaler()
            df[col]=scalers[col].fit_transform(df[col].values.reshape(-1, 1))
    return scalers,df,list_of_col_log_scaled
def process_data_numeric_with_existing_scalers_xgb(df,scalers,list_of_col_log_scaled):    
    if(len(scalers.keys())==0):
        return df
    for col in df.columns:
        df[col]=scalers[col].transform(df[col].values.reshape(-1, 1))
    return df

In [ ]:
def feature_extraction_xgb(df,actual,kanwil,segmen,produk,level='ro'):
    df_dump=pd.DataFrame()
    if(level=='nas'):
        top_15_cor = df[df.columns].corrwith(actual).abs().sort_values(ascending=False).head(15)
        df_dump = df[top_15_cor.index.tolist()]
        top_15_cor.to_excel(f"../results/v5/pinjaman/prophet_xgb/correlation_analysis/{kanwil}_{segmen}_{produk}.xlsx")

        return df_dump
    elif(level=='ro'):
        ro_col=static_columns_pinjaman_merchant+static_columns_rm+static_columns_pinjaman_rasio_fin_ro_level+static_columns_pinjaman_makro_ro_level
        nas_col=static_columns_pinjaman_makro_nas_level

        ro_col_filtered = [col for col in ro_col if col in df.columns]
        nas_col_filtered = [col for col in nas_col if col in df.columns]
        if not ro_col_filtered and not nas_col_filtered:
            return df
        top_15_cor = pd.Series()  # Default empty series
        above_average_corr_columns = pd.Series()  # Default empty series

        if ro_col_filtered or nas_col_filtered:
            if nas_col_filtered:
                top_15_cor = df[nas_col_filtered].corrwith(actual).abs().sort_values(ascending=False).head(15)
            
            if ro_col_filtered:
                above_average_corr = df[ro_col_filtered].corrwith(actual).abs().sort_values(ascending=False)
                if(len(above_average_corr[above_average_corr > 0.6])==0):
                    above_average_corr_columns=above_average_corr.head(5)
                else:
                    above_average_corr_columns = above_average_corr[above_average_corr > 0.6]
        df = df[top_15_cor.index.tolist() + above_average_corr_columns.index.tolist()]
        combined_df = pd.concat([top_15_cor, above_average_corr_columns]).to_frame(name="Correlation")
        combined_df.to_excel(f"../results/v5/pinjaman/prophet_xgb/correlation_analysis/{kanwil}_{segmen}_{produk}.xlsx")
        return df

In [ ]:
def calculate_feature_importance_in_percentage(feature_importances):
    total_importance = sum(feature_importances.values())
    feature_importances_percent = {feature: (importance / total_importance) * 100 for feature, importance in feature_importances.items()}
    df_feature_importances_percent = pd.DataFrame(
        list(feature_importances_percent.items()), 
        columns=['Feature', 'Importance (%)']
    )
    df_feature_importances_percent = df_feature_importances_percent.sort_values(by='Importance (%)', ascending=False)    
    return df_feature_importances_percent

In [ ]:
all_data_prophet = pd.read_excel('../results/v5/pinjaman/prophet_xgb/final_results_prophet/all_results_pinjaman.xlsx', index_col=None)
combinations = all_data_prophet[['Kanwil', 'Segmen', 'Product']].drop_duplicates()
combinations = list(combinations.itertuples(index=False, name=None))

In [ ]:
df_all = df_all.rename(columns={'periode': 'ds', 'kanwil': 'Kanwil','segmen':'Segmen','produk':'Product'})
merged_df = pd.merge(all_data_prophet, df_all, how='inner', on=['ds', 'Kanwil', 'Segmen', 'Product'])
merged_df_xgb=merged_df.drop(columns=['amount'])
merged_df_xgb['residu'] = merged_df_xgb['Amount'] - merged_df_xgb['Amount_Predict']

In [ ]:
all_xgb_results=pd.DataFrame()
for comb in combinations:
    print(f"Processing {comb}")
    merged_df_xgb_test=merged_df_xgb.loc[(merged_df_xgb['Kanwil']==comb[0])&(merged_df_xgb['Product']==comb[2])&(merged_df_xgb['Segmen']==comb[1])]
    train=merged_df_xgb_test.loc[merged_df_xgb_test['ds']<='2024-03-01']
    test=merged_df_xgb_test.loc[(merged_df_xgb_test['ds']>'2024-03-01')&(merged_df_xgb_test['ds']<='2024-06-01')]
    forecast=merged_df_xgb_test.loc[(merged_df_xgb_test['ds']>'2024-06-01')]

    X = train.drop(columns=['residu','Kanwil','Amount','Amount_Predict','Segmen','Product','Partition','Method'])
    y = train['residu']
    X['year'] = X['ds'].dt.year
    X['month'] = X['ds'].dt.month
    X = X.drop(columns=['ds'])

    X_test = test.drop(columns=['residu','Kanwil','Amount','Amount_Predict','Segmen','Product','Partition','Method'])
    y_test = test['residu']
    X_test['year'] = X_test['ds'].dt.year
    X_test['month'] = X_test['ds'].dt.month
    X_test = X_test.drop(columns=['ds'])

    X_forecast = forecast.drop(columns=['residu','Kanwil','Amount','Amount_Predict','Segmen','Product','Partition','Method'])
    y_forecast = forecast['residu']
    X_forecast['year'] = X_forecast['ds'].dt.year
    X_forecast['month'] = X_forecast['ds'].dt.month
    X_forecast = X_forecast.drop(columns=['ds'])
    
    scalers_train,X_scaled,list_of_col_log_scaled_train=process_data_numeric_xgb(X,'Robust')
    X_test_scaled=process_data_numeric_with_existing_scalers_xgb(X_test,scalers_train,list_of_col_log_scaled_train)
    X_forecast_scaled=process_data_numeric_with_existing_scalers_xgb(X_forecast,scalers_train,list_of_col_log_scaled_train)

    if(comb[0]=='NASIONAL'):
        X_train_scaled_extract=feature_extraction_xgb(X_scaled.reset_index(drop=True),pd.Series(y),comb[0],comb[1],comb[2],'nas').reset_index(drop=True)
    else:
        X_train_scaled_extract=feature_extraction_xgb(X_scaled.reset_index(drop=True),pd.Series(y),comb[0],comb[1],comb[2],'ro').reset_index(drop=True)

    X_test_scaled_extract=X_test_scaled[X_train_scaled_extract.columns].reset_index(drop=True)
    X_forecast_scaled_extract=X_forecast_scaled[X_train_scaled_extract.columns].reset_index(drop=True)


    scalers_train['residu']=RobustScaler()
    y=scalers_train['residu'].fit_transform(y.values.reshape(-1, 1)).flatten()
    y_test=scalers_train['residu'].transform(y_test.values.reshape(-1, 1)).flatten()
    y_forecast=scalers_train['residu'].transform(y_forecast.values.reshape(-1, 1)).flatten()

    model = XGBRegressor(objective='reg:squarederror', n_estimators=100, enable_categorical=True)

    param_grid = {
            'colsample_bytree': [0.5],
            'gamma': [0.1],
            'learning_rate': [0.01],
            'max_depth': [6],
            'min_child_weight': [3],
            'n_estimators': [100],
            'reg_alpha': [0.5],
            'reg_lambda': [0.5],
            'subsample': [0.6],
            'seed': [42],
    }
    grid_search = GridSearchCV(estimator=model, param_grid=param_grid, 
                            scoring='neg_root_mean_squared_error', 
                            cv=3, n_jobs=-1, verbose=1)

    grid_search.fit(X_train_scaled_extract, y)

    print("Best parameters found: ", grid_search.best_params_)
    print("Best RMSE score: ", -grid_search.best_score_)

    best_model = grid_search.best_estimator_


    y_pred = best_model.predict(X_train_scaled_extract)
    residuals_non_scaled = scalers_train['residu'].inverse_transform(y.reshape(-1, 1)).flatten()
    predictions_non_scaled = scalers_train['residu'].inverse_transform(y_pred.reshape(-1, 1)).flatten()
    train['predictions_xgb']=predictions_non_scaled
    train['Final_Predictions']=train['Amount_Predict']+train['predictions_xgb']

    rmse_scaled = mean_squared_error(y, y_pred, squared=False)
    rmse_non_scaled = mean_squared_error(residuals_non_scaled, predictions_non_scaled, squared=False)

    print(f"Root Mean Squared Error (Scaled Train): {rmse_scaled}")
    print(f"Root Mean Squared Error (Non Scaled Train): {rmse_non_scaled}")

    y_pred_test = best_model.predict(X_test_scaled_extract)
    residuals_non_scaled = scalers_train['residu'].inverse_transform(y_test.reshape(-1, 1)).flatten()
    predictions_non_scaled = scalers_train['residu'].inverse_transform(y_pred_test.reshape(-1, 1)).flatten()
    test['predictions_xgb']=predictions_non_scaled
    test['Final_Predictions']=test['Amount_Predict']+test['predictions_xgb']
    rmse_scaled = mean_squared_error(y_test, y_pred_test, squared=False)
    rmse_non_scaled = mean_squared_error(residuals_non_scaled, predictions_non_scaled, squared=False)
    print(f"Root Mean Squared Error (Scaled Test): {rmse_scaled}")
    print(f"Root Mean Squared Error (Non Scaled Test): {rmse_non_scaled}")

    y_pred_forecast = best_model.predict(X_forecast_scaled_extract)
    residuals_non_scaled = scalers_train['residu'].inverse_transform(y_forecast.reshape(-1, 1)).flatten()
    predictions_non_scaled = scalers_train['residu'].inverse_transform(y_pred_forecast.reshape(-1, 1)).flatten()
    forecast['predictions_xgb']=predictions_non_scaled
    forecast['Final_Predictions']=forecast['Amount_Predict']+forecast['predictions_xgb']


    combined_forecast_xgb = pd.concat([train, test,forecast])
     
    plt.figure(figsize=(10, 6))
    plt.plot(combined_forecast_xgb['ds'], combined_forecast_xgb['Amount'], label='Actual', marker='o')
    plt.plot(combined_forecast_xgb['ds'], combined_forecast_xgb['Final_Predictions'], label='Predicted', marker='x')
    plt.xlabel('Date')
    plt.ylabel('Values')
    plt.title(f'Actual vs Predicted for {comb[0]} - {comb[1]} - {comb[2]}')
    plt.legend()    
    plt.tight_layout()
    plot_filename = f"../visualization/new/pinjaman/prophet_xgb/plot_{comb[0]}_{comb[1]}_{comb[2]}.png"
    plt.savefig(plot_filename)
    plt.show()
    print(f"Plot saved as {plot_filename}")


    feature_importances = best_model.get_booster().get_score(importance_type='weight')
    df_feature_importances = pd.DataFrame(
        list(feature_importances.items()), 
        columns=['Feature', 'Importance']
    )
    df_feature_importances = df_feature_importances.sort_values(by='Importance', ascending=False)
    df_feature_importances_percent = calculate_feature_importance_in_percentage(feature_importances).reset_index(drop=True).to_excel(f'../results/v5/pinjaman/prophet_xgb/feature_importances/{comb[0]}_{comb[1]}_{comb[2]}.xlsx')
    all_xgb_results=pd.concat([all_xgb_results,combined_forecast_xgb])
    dump_model(model,f'../results/v5/pinjaman/prophet_xgb/dump_model/xgb/{comb[0]}_{comb[1]}_{comb[2]}.pkl',scalers_train,None,None,None)
all_xgb_results.reset_index(drop=True).to_excel(f'../results/v5/pinjaman/prophet_xgb/final_results_prophet_xgb/all_final_results_pinjaman.xlsx')

# Replot

In [ ]:
all_xgb_results_new=pd.read_excel(f'../results/v5/pinjaman/prophet_xgb/final_results_prophet_xgb/all_final_results_pinjaman.xlsx')

In [ ]:
combined_forecast_xgb=all_xgb_results_new.loc[(all_xgb_results_new['Kanwil']==comb[0])&(all_xgb_results_new['Segmen']==comb[1])&(all_xgb_results_new['Product']==comb[2])]     
combined_forecast_xgb

In [ ]:
for comb in combinations:
    combined_forecast_xgb=all_xgb_results_new.loc[(all_xgb_results_new['Kanwil']==comb[0])&(all_xgb_results_new['Segmen']==comb[1])&(all_xgb_results_new['Product']==comb[2])]     
    combined_forecast_xgb=combined_forecast_xgb.sort_values(by='ds')
    plt.figure(figsize=(10, 6))
    plt.plot(combined_forecast_xgb['ds'], combined_forecast_xgb['Amount'], label='Actual', marker='o')
    plt.plot(combined_forecast_xgb['ds'], combined_forecast_xgb['Final_Predictions'], label='Predicted', marker='x')
    plt.xlabel('Date')
    plt.ylabel('Values')
    plt.title(f'Actual vs Predicted for {comb[0]} - {comb[1]} - {comb[2]}')
    plt.legend()    
    plt.tight_layout()
    plot_filename = f"../visualization/new/pinjaman/prophet_xgb/plot_{comb[0]}_{comb[1]}_{comb[2]}.png"
    plt.savefig(plot_filename)
    plt.show()
    print(f"Plot saved as {plot_filename}")